In [4]:
%reload_ext autoreload
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
import IPython.display as ipd
import whisper
import sys
sys.path.append("/home/romolo/VT1/coqui-tts")
from model_conf import ModelPaths, load_tts_and_trainer

/home/romolo/VT1/coqui-tts/.venv/lib/python3.11/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/home/romolo/VT1/coqui-tts/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/romolo/VT1/coqui-tts/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


In [6]:
paths = ModelPaths()
tts, model, train_model, config = load_tts_and_trainer(paths)

 > Using model: xtts
>> DVAE weights restored from: /home/romolo/VT1/coqui-tts/XTTS_v2.0_original_model_files/dvae.pth


In [7]:
import torchaudio

def crop_audio(input_path, output_path, duration_sec=5):
    waveform, sample_rate = torchaudio.load(input_path)
    num_samples = int(duration_sec * sample_rate)
    cropped_waveform = waveform[:, :num_samples]
    torchaudio.save(output_path, cropped_waveform, sample_rate)

# Example usage:
crop_audio("/home/romolo/VT1/coqui-tts/data/target_no_sr.wav", "/home/romolo/VT1/coqui-tts/data/TARGET_5s.wav", duration_sec=5)


In [8]:
import torch
import torchaudio
import noisereduce as nr
import numpy as np
import librosa
import soundfile as sf

def denoise_audio(input_path, output_path):
    # Load audio
    waveform, sample_rate = torchaudio.load(input_path)

    # Convert to mono if stereo
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    audio_np = waveform.squeeze(0).numpy().astype(np.float32)

    # --- Trim silence using librosa ---
    trimmed, _ = librosa.effects.trim(audio_np, top_db=50)
    audio_np = trimmed.astype(np.float32)

    # --- Apply light denoising ---
    reduced_noise = nr.reduce_noise(
        y=audio_np,
        sr=sample_rate,
        stationary=False,
        prop_decrease=0.7,
        n_fft=1024
    )

    # Normalize
    reduced_noise = reduced_noise / np.max(np.abs(reduced_noise) + 1e-6)

    # --- Save with torchaudio or soundfile ---
    sf.write(output_path, reduced_noise, sample_rate)
    print(f"Denoised audio saved to: {output_path}")

denoise_audio(
    "/home/romolo/VT1/coqui-tts/data/target.wav",
    "/home/romolo/VT1/coqui-tts/data/TARGET_no_noise.wav"
)

Denoised audio saved to: /home/romolo/VT1/coqui-tts/data/TARGET_no_noise.wav


In [9]:
asr_model = whisper.load_model("base")

In [10]:
import os

In [11]:
ref_samples = os.listdir("/home/romolo/VT1/coqui-tts/test_data/Dataset/references/26/")
ref_samples = ["/home/romolo/VT1/coqui-tts/test_data/Dataset/references/26/" + i for i in ref_samples]

In [12]:
orig_target_sample = "/home/romolo/VT1/coqui-tts/data/target.wav"
text = asr_model.transcribe(orig_target_sample)['text']

# split and stitch audio

In [13]:
def split_audio(audio, segment_length_samples):
    segments = []
    for start in range(0, audio.shape[-1], segment_length_samples):
        end = min(start + segment_length_samples, audio.shape[-1])
        segments.append(audio[..., start:end])
    return segments

def process_segments(segments, scoring_fn, threshold):
    accepted = []
    to_improve = []
    for i, seg in enumerate(segments):
        score = scoring_fn(seg)
        if score > threshold:
            accepted.append(seg)
        else:
            to_improve.append((i, seg))
    return accepted, to_improve

def stitch_segments(segments):
    # Convert all segments to tensors if they are numpy arrays
    segments = [torch.from_numpy(seg) if isinstance(seg, np.ndarray) else seg for seg in segments]
    return torch.cat(segments, dim=-1)

In [14]:
from TTS.tts.models.xtts import load_audio
import soundfile as sf

In [15]:
import soundfile as sf

audio = load_audio(orig_target_sample, 22050)
segment_length = 22050 * 2  # 2 seconds per segment
segments = split_audio(audio, segment_length)

for i, segment in enumerate(segments):
    segment_np = segment.detach().cpu().numpy().astype('float32')
    # If shape is [1, samples], squeeze to [samples]
    if segment_np.ndim == 2 and segment_np.shape[0] == 1:
        segment_np = segment_np.squeeze(0)
    # Remove NaNs/Infs if any
    segment_np = np.nan_to_num(segment_np)
    sf.write(f"/home/romolo/VT1/coqui-tts/data/outputs/process/audio_{i}.wav", segment_np, 22050)

In [16]:
sorted(os.listdir(f"/home/romolo/VT1/coqui-tts/data/outputs/process/"))

['audio_0.wav',
 'audio_1.wav',
 'audio_2.wav',
 'audio_3.wav',
 'audio_4.wav',
 'audio_5.wav',
 'audio_6.wav']

In [17]:
"""
wavs = []
for file in sorted(os.listdir(f"/home/romolo/VT1/coqui-tts/data/outputs/process/")):
    path = f"/home/romolo/VT1/coqui-tts/data/outputs/process/" + file
    text = asr_model.transcribe(path)['text']
    wav = model.forward_from_audios_and_text('en',text,path,ref_samples[0],train_model,config.model_args.max_conditioning_length, config.model_args.min_conditioning_length)
    wavs.append(wav['wav'])"""

'\nwavs = []\nfor file in sorted(os.listdir(f"/home/romolo/VT1/coqui-tts/data/outputs/process/")):\n    path = f"/home/romolo/VT1/coqui-tts/data/outputs/process/" + file\n    text = asr_model.transcribe(path)[\'text\']\n    wav = model.forward_from_audios_and_text(\'en\',text,path,ref_samples[0],train_model,config.model_args.max_conditioning_length, config.model_args.min_conditioning_length)\n    wavs.append(wav[\'wav\'])'

In [18]:
"""audio = stitch_segments(wavs)"""

'audio = stitch_segments(wavs)'

In [19]:
"""ipd.Audio(audio, rate=24000)"""

'ipd.Audio(audio, rate=24000)'

In [20]:
"""target_sample = orig_target_sample
for i in range(0,10):
    wav = model.forward_from_audios_and_text('en',text,target_sample,ref_samples[0],train_model,config.model_args.max_conditioning_length, config.model_args.min_conditioning_length)
    tts.synthesizer.save_wav(wav=wav['wav'], path=f"/home/romolo/VT1/coqui-tts/data/outputs/iterate_output_{i}.wav")
    target_sample = f"/home/romolo/VT1/coqui-tts/data/outputs/iterate_output_{i}.wav"
    """

'target_sample = orig_target_sample\nfor i in range(0,10):\n    wav = model.forward_from_audios_and_text(\'en\',text,target_sample,ref_samples[0],train_model,config.model_args.max_conditioning_length, config.model_args.min_conditioning_length)\n    tts.synthesizer.save_wav(wav=wav[\'wav\'], path=f"/home/romolo/VT1/coqui-tts/data/outputs/iterate_output_{i}.wav")\n    target_sample = f"/home/romolo/VT1/coqui-tts/data/outputs/iterate_output_{i}.wav"\n    '

In [21]:
"""ipd.Audio(wav['wav'], rate=24000)"""

"ipd.Audio(wav['wav'], rate=24000)"

In [22]:
from huggingface_hub import hf_hub_download

# automatically checks for cached file, optionally set `cache_dir` location
model_file = hf_hub_download(repo_id='Jenthe/ECAPA2', filename='ecapa2.pt', cache_dir=None)

In [23]:
import torch
import torchaudio
import torch.nn.functional as F

ecapa2 = torch.jit.load(model_file, map_location='cuda')


In [24]:
paths = ModelPaths()
tts, model, train_model, config = load_tts_and_trainer(paths)

 > Using model: xtts
>> DVAE weights restored from: /home/romolo/VT1/coqui-tts/XTTS_v2.0_original_model_files/dvae.pth


In [25]:
output = model.forward_iteration('en',text,orig_target_sample,ref_samples[0],train_model,config.model_args.max_conditioning_length, config.model_args.min_conditioning_length,tts,ecapa2,asr_model)

Iteration 0: BLUE: 0.7976501109363606, WER: 0.08333333333333333, Target Cosine: tensor([0.1321], device='cuda:0'), Reference Cosine: tensor([0.5890], device='cuda:0')
Iteration 1: BLUE: 0.7250357303325181, WER: 0.1388888888888889, Target Cosine: tensor([0.1854], device='cuda:0'), Reference Cosine: tensor([0.6718], device='cuda:0')
Iteration 2: BLUE: 0.5672479285726278, WER: 0.2222222222222222, Target Cosine: tensor([0.2039], device='cuda:0'), Reference Cosine: tensor([0.6945], device='cuda:0')
Iteration 3: BLUE: 0.49413447564643725, WER: 0.25, Target Cosine: tensor([0.1917], device='cuda:0'), Reference Cosine: tensor([0.6693], device='cuda:0')
Iteration 4: BLUE: 0.6388087677672988, WER: 0.19444444444444445, Target Cosine: tensor([0.1758], device='cuda:0'), Reference Cosine: tensor([0.7084], device='cuda:0')
Iteration 5: BLUE: 0.6406043419738114, WER: 0.16666666666666666, Target Cosine: tensor([0.1740], device='cuda:0'), Reference Cosine: tensor([0.7053], device='cuda:0')
Iteration 6: B

In [26]:
from run_test import segment_quality_score

In [27]:
tar_audio, sr = torchaudio.load(orig_target_sample) # sample rate of 16 kHz expected
tar_audio = torchaudio.functional.resample(tar_audio, orig_freq=sr, new_freq=16_000)
tar_embedding = ecapa2(tar_audio.to('cuda'))

ref_audio, sr2 = torchaudio.load(ref_samples[0]) # sample rate of 16 kHz expected
ref_audio = torchaudio.functional.resample(ref_audio, orig_freq=sr2, new_freq=16_000)
ref_embedding = ecapa2(ref_audio.to('cuda'))

In [28]:
scores, info = segment_quality_score(asr_model, tar_embedding, ref_embedding,output[3][0],ecapa2)

In [29]:
info

{0: {'overall_quality': 0.6807610243558884,
  'wer': 0.0,
  'blue': 1.0,
  'target_sim': 0.17274877429008484,
  'ref_sim': 0.5543094277381897},
 1: {'overall_quality': 0.7235190659761428,
  'wer': 0.0,
  'blue': 1.0,
  'target_sim': 0.13204017281532288,
  'ref_sim': 0.6052185297012329},
 2: {'overall_quality': 0.5364039093255997,
  'wer': 0.0,
  'blue': 1.0,
  'target_sim': 0.016952872276306152,
  'ref_sim': 0.23581628501415253},
 3: {'overall_quality': 0.6451777957379818,
  'wer': 0.0,
  'blue': 1.0,
  'target_sim': 0.036745838820934296,
  'ref_sim': 0.42700257897377014},
 4: {'overall_quality': 0.5441011407012055,
  'wer': 0.0,
  'blue': 0.5757197301274735,
  'target_sim': 0.31493881344795227,
  'ref_sim': 0.43299466371536255},
 5: {'overall_quality': 0.6460348576307298,
  'wer': 0.0,
  'blue': 1.0,
  'target_sim': 0.12714800238609314,
  'ref_sim': 0.4736320972442627},
 6: {'overall_quality': 0.5701914236480305,
  'wer': 0.0,
  'blue': 0.5757197301274735,
  'target_sim': 0.0437160208

In [ ]:
def calc_sim(aud1,aud2):
    #cossim between embeddings
    audio, sr = torchaudio.load(aud1)
    audio = torchaudio.functional.resample(audio, orig_freq=sr, new_freq=16_000)# sample rate of 16 kHz expected
    embedding = ecapa2(audio.to('cuda'))
    ref_audio, sr = torchaudio.load(aud2) # sample rate of 16 kHz expected
    ref_audio = torchaudio.functional.resample(ref_audio, orig_freq=sr, new_freq=16_000)
    ref_embedding = ecapa2(ref_audio.to('cuda'))
    sim = F.cosine_similarity(embedding, ref_embedding)
    return sim

In [ ]:
for i in range(0,10):
    targ = calc_sim(f"/home/romolo/VT1/coqui-tts/data/outputs/iterate_output_{i}.wav","/home/romolo/VT1/coqui-tts/data/target.wav")
    print(f"sample {i}")
    print(f"sim -- target to output: {targ}")
    ref = calc_sim(f"/home/romolo/VT1/coqui-tts/data/outputs/iterate_output_{i}.wav",ref_samples[0])
    print(f"sim -- ref to output: {ref}")


RuntimeError: Failed to open the input "/home/romolo/VT1/coqui-tts/data/outputs/iterate_output_0.wav" (No such file or directory).
Exception raised from get_input_format_context at /__w/audio/audio/pytorch/audio/src/libtorio/ffmpeg/stream_reader/stream_reader.cpp:42 (most recent call first):
frame #0: c10::Error::Error(c10::SourceLocation, std::string) + 0x96 (0x7fae9cf6c1b6 in /home/romolo/VT1/coqui-tts/.venv/lib/python3.11/site-packages/torch/lib/libc10.so)
frame #1: c10::detail::torchCheckFail(char const*, char const*, unsigned int, std::string const&) + 0x64 (0x7fae9cf15a76 in /home/romolo/VT1/coqui-tts/.venv/lib/python3.11/site-packages/torch/lib/libc10.so)
frame #2: <unknown function> + 0x42034 (0x7fad28d8c034 in /home/romolo/VT1/coqui-tts/.venv/lib/python3.11/site-packages/torio/lib/libtorio_ffmpeg4.so)
frame #3: torio::io::StreamingMediaDecoder::StreamingMediaDecoder(std::string const&, std::optional<std::string> const&, std::optional<std::map<std::string, std::string, std::les